# TP 2-1 : Topic Modeling et Prétraitement — Dataset IMDB

**Objectifs :**
- Maîtriser le prétraitement d'un corpus textuel (nettoyage, normalisation, tokenisation).
- Implémenter et interpréter un modèle LDA (Latent Dirichlet Allocation).
- Comparer LDA avec la factorisation par matrices non-négatives (NMF).
- Évaluer l'impact du prétraitement sur l'interprétabilité des topics.


## Installation et imports

In [ ]:
!pip install pandas numpy scikit-learn nltk gensim matplotlib wordcloud

import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

from gensim.corpora import Dictionary
from gensim.models import LdaModel

# Téléchargement des ressources NLTK nécessaires (une seule fois)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\MOUNET_ANDRE_ARMEL_R\AppData\Roaming\nltk_dat
[nltk_data]     a...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\MOUNET_ANDRE_ARMEL_R\AppData\Roaming\nltk_dat
[nltk_data]     a...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\MOUNET_ANDRE_ARMEL_R\AppData\Roaming\nltk_dat
[nltk_data]     a...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

---
## Partie A : Prétraitement

### A.1 — Chargement du dataset et exploration initiale

**Objectif :** charger le dataset et vérifier sa structure avant tout traitement, comme demandé dans les consignes (analyse des données avant NLP).

In [1]:
# Chargement du dataset
df = pd.read_csv("../data/IMDBDataset.csv")

# Aperçu des 10 premières lignes
print(df.head(10))

print("\nDimensions du dataset :", df.shape)
print("\nColonnes :", df.columns.tolist())

# Vérification des valeurs manquantes et des doublons
print("\nValeurs manquantes par colonne :\n", df.isnull().sum())
print("\nNombre de doublons :", df.duplicated().sum())

# Distribution des labels (positive / negative)
print("\nDistribution des labels (effectifs) :")
print(df["sentiment"].value_counts())
print("\nDistribution des labels (pourcentages) :")
print(df["sentiment"].value_counts(normalize=True) * 100)

df["sentiment"].value_counts().plot(kind="bar", color=["seagreen", "indianred"])
plt.title("Distribution des labels (positive / negative)")
plt.xlabel("Sentiment")
plt.ylabel("Nombre de critiques")
plt.xticks(rotation=0)
plt.show()

NameError: name 'pd' is not defined

**Résultats obtenus :** le dataset contient bien 50 000 lignes et 2 colonnes (`review`, `sentiment`), sans aucune valeur manquante. Les labels sont **parfaitement équilibrés** (25 000 / 25 000, soit 50 %/50 %). En revanche, on observe **418 lignes strictement dupliquées** (texte de critique identique). Ces doublons ne sont pas supprimés pour rester fidèle aux consignes (aucune étape de dé-duplication n'est demandée explicitement), mais ils représentent moins de 1 % du corpus : leur impact sur les topics obtenus en Parties B, C, D reste donc négligeable.

### A.2 — Fonction de prétraitement `preprocess_text`

**Objectif :** construire une fonction qui nettoie et normalise un texte brut selon les consignes :
- minuscules, suppression des balises HTML, suppression de la ponctuation/caractères spéciaux,
- tokenisation, suppression des stopwords (NLTK anglais), lemmatisation (`WordNetLemmatizer`).

**Pourquoi ces étapes ?** Les critiques IMDB contiennent du HTML résiduel (`<br />`), de la ponctuation qui n'apporte pas d'information thématique, et des mots très fréquents (stopwords : *the, is, and...*) qui polluent les topics s'ils ne sont pas retirés. La lemmatisation regroupe les variantes d'un même mot (*acting, acted, acts* → *act*), ce qui réduit la taille du vocabulaire et améliore la cohérence des topics.

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # 1. Conversion en minuscules
    text = text.lower()
    # 2. Suppression des balises HTML (ex: <br />)
    text = re.sub(r"<.*?>", " ", text)
    # 3. Suppression des caractères spéciaux et de la ponctuation (on ne garde que les lettres)
    text = re.sub(r"[^a-z\s]", " ", text)
    # 4. Tokenisation simple
    tokens = text.split()
    # 5. Suppression des stopwords + mots trop courts (bruit résiduel)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    # 6. Lemmatisation
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

**Explication simple du code :**
- `re.sub(r"<.*?>", " ", text)` remplace toute balise HTML (`<br />`, `<i>`, etc.) par un espace.
- `re.sub(r"[^a-z\s]", " ", text)` ne garde que les lettres minuscules et les espaces (chiffres, ponctuation, symboles supprimés).
- `text.split()` découpe la chaîne en mots (tokenisation basique par espaces).
- La liste en compréhension retire les stopwords et les tokens de moins de 3 lettres (bruit résiduel : "a", "an", "ok"...).
- `lemmatizer.lemmatize(t)` ramène chaque mot à sa forme canonique (lemme).

### A.3 — Application au dataset

**Objectif :** appliquer `preprocess_text` à la colonne `review` et stocker le résultat dans `review_clean`.

In [ ]:
df["review_clean"] = df["review"].apply(preprocess_text)

# Exemple avant / après sur la première critique
print("AVANT :\n", df["review"].iloc[0])
print("\nAPRÈS :\n", df["review_clean"].iloc[0])

AVANT :
 One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due t

**Résultat obtenu :** le texte « après » ne contient plus de balises HTML ni de ponctuation, uniquement des mots en minuscules, lemmatisés, sans stopwords — nettement plus court que le texte original (le texte d'exemple passe d'environ 1500 caractères à ~950 caractères une fois nettoyé).

### A.4 — Vectorisation (Bag of Words)

**Objectif :** transformer le corpus prétraité en matrice document-terme avec `CountVectorizer`.

**Pourquoi ces paramètres ?** `max_features=5000` limite le vocabulaire aux 5000 mots les plus fréquents (contrôle la dimensionnalité) ; `max_df=0.95` élimine les mots présents dans plus de 95% des documents (trop génériques pour être discriminants) ; `min_df=2` élimine les mots trop rares (bruit, fautes de frappe) présents dans moins de 2 documents.

In [ ]:
count_vectorizer = CountVectorizer(max_features=5000, max_df=0.95, min_df=2)
dtm = count_vectorizer.fit_transform(df["review_clean"])

print("Forme de la matrice document-terme (Bag of Words) :", dtm.shape)

Forme de la matrice document-terme (Bag of Words) : (50000, 5000)


**Résultat obtenu :** la matrice document-terme a la forme `(50000, 5000)` : 50 000 documents (critiques) en lignes, 5 000 termes (le vocabulaire maximal fixé) en colonnes.

---
## Partie B : LDA avec scikit-learn

### B.1-B.2 — Instanciation et entraînement du modèle

**Paramètres imposés :** `n_components=10` (on utilise `n_components`, le nom actuel du paramètre dans scikit-learn — `n_topics` a été renommé il y a plusieurs versions), `learning_method='online'`, `random_state=42`, `max_iter=10`.

In [ ]:
lda_model = LatentDirichletAllocation(
    n_components=10,
    learning_method="online",
    random_state=42,
    max_iter=10,
)
lda_model.fit(dtm)
print("Modèle LDA entraîné.")

### B.3 — Fonction d'affichage des topics

**Objectif :** afficher, pour chaque topic, les mots les plus représentatifs (poids les plus élevés dans `model.components_`).

In [ ]:
def display_topics(model, feature_names, no_top_words):
    for topic_idx, topic in enumerate(model.components_):
        top_words_idx = topic.argsort()[:-no_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_words_idx]
        print(f"Topic #{topic_idx}: {' '.join(top_words)}")

feature_names_cv = count_vectorizer.get_feature_names_out()
print("=== Topics LDA (scikit-learn) ===")
display_topics(lda_model, feature_names_cv, 10)

### B.4 — Interprétation des topics obtenus

| Topic | Mots dominants | Thème proposé |
|---|---|---|
| #0 | guy, car, school, high, team | Vie quotidienne / cadre scolaire (peu cohérent) |
| #1 | movie, like, good, bad, really | Vocabulaire évaluatif général (pas un vrai thème) |
| #2 | john, black, richard, robert, western | Noms propres / western — mélange de personnages |
| #3 | film, character, life, story, people | Discussion narrative générique |
| #4 | show, funny, series, comedy, episode | Séries TV / comédie |
| #5 | film, horror, effect, budget, low | Horreur à petit budget / effets spéciaux |
| #6 | man, woman, girl, house, killer, murder | Thriller / polar (meurtre) |
| #7 | role, performance, play, actor, cast | Performance d'acteurs |
| #8 | film, movie, great, story, year, seen | Vocabulaire évaluatif positif générique |
| #9 | war, american, world, soldier, battle | Guerre / film historique |

**Constat :** seuls quelques topics sont clairement interprétables comme des *genres* (#4 séries/comédie, #5 horreur, #6 thriller, #9 guerre, #7 performance d'acteurs). Les topics #1, #3 et #8 se recoupent fortement : ils regroupent surtout le vocabulaire évaluatif générique des critiques (*movie, like, good, story, great*...) plutôt qu'un thème de contenu. C'est cohérent avec le fait que ce corpus est composé de critiques (le vocabulaire de jugement/évaluation domine autant que le vocabulaire de genre).

### B.5 — Perplexité du modèle

**Objectif :** évaluer la qualité du modèle LDA avec la perplexité, une mesure de la capacité du modèle à « prédire » le corpus d'entraînement (plus elle est basse, mieux le modèle explique les données — mais elle ne mesure pas directement l'interprétabilité humaine des topics).

In [ ]:
perplexity = lda_model.perplexity(dtm)
print("Perplexité du modèle LDA sur le corpus d'entraînement :", perplexity)

Perplexité du modèle LDA sur le corpus d'entraînement : 1703.0177601136702


**Résultat obtenu :** perplexité ≈ **1703**. Cette valeur seule n'est interprétable qu'en comparaison (avec un autre nombre de topics ou un autre prétraitement) : plus elle diminue quand on ajuste les hyperparamètres, mieux le modèle explique statistiquement le corpus — mais une perplexité basse n'implique pas forcément des topics plus interprétables par un humain (c'est justement ce qu'on observe en comparant avec NMF ci-dessous).

---
## Partie C : NMF et comparaison

### C.1 — Vectorisation TF-IDF

**Objectif :** vectoriser le même corpus prétraité avec `TfidfVectorizer` (mêmes contraintes que pour CountVectorizer), car NMF fonctionne généralement mieux sur une représentation TF-IDF que sur des comptes bruts.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000, max_df=0.95, min_df=2)
dtm_tfidf = tfidf_vectorizer.fit_transform(df["review_clean"])

print("Forme de la matrice TF-IDF :", dtm_tfidf.shape)

Forme de la matrice TF-IDF : (50000, 5000)


### C.2-C.3 — Modèle NMF

**Pourquoi ces paramètres ?** `n_components=10` pour rester comparable au LDA ; `max_iter=500` car NMF converge par optimisation itérative (contrairement à LDA qui est probabiliste) et peut nécessiter plus d'itérations pour converger correctement.

In [ ]:
nmf_model = NMF(n_components=10, random_state=42, max_iter=500)
nmf_model.fit(dtm_tfidf)

feature_names_tfidf = tfidf_vectorizer.get_feature_names_out()
print("=== Topics NMF (scikit-learn) ===")
display_topics(nmf_model, feature_names_tfidf, 10)

=== Topics NMF (scikit-learn) ===
Topic #0: life man woman character story one two young scene family
Topic #1: movie watch made make watching see recommend scene want enjoy
Topic #2: show episode series season television watch funny character family watching
Topic #3: bad acting worst even plot terrible awful script actor good
Topic #4: film made director make making many cinema see festival scene
Topic #5: great good actor story well performance role character best cast
Topic #6: book read novel story version character series adaptation original based
Topic #7: one time seen ever year first saw dvd see best
Topic #8: horror zombie gore effect budget flick low monster killer one
Topic #9: like really people think get would thing know funny say


### C.4 — Comparaison qualitative LDA vs NMF

**Observations concrètes sur nos résultats :**
- Les topics NMF sont nettement plus **nets et disjoints** que ceux de LDA : #2 (séries TV), #6 (adaptations littéraires), #8 (horreur/zombie/gore) et #4 (réalisation/festival de cinéma) sont des thèmes propres qui n'apparaissaient pas aussi clairement avec LDA.
- LDA produit plusieurs topics redondants centrés sur le vocabulaire évaluatif générique (#1, #3, #8 vus en partie B) ; NMF isole ce vocabulaire plus proprement dans un seul topic clair (#9 : *like, really, people, think* — registre familier/opinion) et un topic dédié aux critiques négatives (#3 : *bad, worst, terrible, awful*).
- NMF fait apparaître un topic totalement absent de la sortie LDA : les **adaptations de livres** (#6 : *book, read, novel, adaptation, based*) — un vrai sous-thème du corpus IMDB que LDA n'a pas isolé avec ces réglages.
- Dans l'ensemble, **NMF semble plus interprétable ici**, ce qui est cohérent avec l'usage d'une entrée TF-IDF (qui atténue déjà les mots trop génériques) combinée à la contrainte de non-négativité qui pousse les topics à être plus « purs » (addition plutôt que compensation entre mots).

### C.5 — Question théorique : LDA vs NMF

**LDA (Latent Dirichlet Allocation)** est un modèle **génératif bayésien** : il suppose un processus probabiliste selon lequel chaque document est généré en tirant une distribution de topics (loi de Dirichlet), puis pour chaque mot du document, un topic est tiré selon cette distribution, puis un mot est tiré selon la distribution de mots de ce topic. LDA produit donc des distributions de probabilité (un document a une probabilité d'appartenir à chaque topic, un topic a une probabilité sur chaque mot), avec une incertitude explicitement modélisée.

**NMF (Non-negative Matrix Factorization)** est une méthode **algébrique/déterministe** : elle factorise directement la matrice document-terme `V` (dimensions documents × mots) en deux matrices non négatives `W` (documents × topics) et `H` (topics × mots) telles que `V ≈ W × H`, en minimisant une erreur de reconstruction (norme de Frobenius par défaut). Il n'y a pas de modèle probabiliste sous-jacent : la contrainte de non-négativité impose que les topics s'additionnent plutôt que se soustraient, ce qui favorise souvent des topics plus « purs » et facilement interprétables — ce que confirme notre comparaison ci-dessus.

**Quand privilégier l'une ou l'autre ?**
- **LDA** est préférable quand on veut un cadre probabiliste rigoureux, une incertitude quantifiable sur l'appartenance aux topics, la possibilité d'appliquer le modèle à de nouveaux documents de façon cohérente (inférence bayésienne), ou quand le corpus est très volumineux (LDA en mode `online` passe à l'échelle facilement, ce que nous avons observé : il reste utilisable sur 50 000 documents).
- **NMF** est préférable quand on cherche des topics plus interprétables/nets sur un corpus de taille modérée, avec une représentation TF-IDF, et quand on n'a pas besoin d'une interprétation probabiliste. Sur nos données, NMF a aussi été bien plus rapide à converger que LDA.

---
## Partie D : Exploration avec Gensim

### D.1-D.2 — Corpus Gensim et dictionnaire filtré

**Objectif :** reformater le corpus prétraité au format attendu par Gensim (liste de listes de tokens), construire un dictionnaire, puis filtrer les extrêmes.

**Pourquoi `filter_extremes` ?** `no_below=100` retire les mots présents dans moins de 100 documents (trop rares pour être des topics robustes sur 50 000 documents) ; `no_above=0.5` retire les mots présents dans plus de 50% des documents (trop génériques). C'est un filtrage plus agressif que celui de CountVectorizer/TfidfVectorizer en Partie A/C, ce qui nous permet d'observer l'impact du prétraitement sur les topics.

In [ ]:
# Liste de listes de tokens (chaque document = liste de mots)
texts = [doc.split() for doc in df["review_clean"]]

# Dictionnaire Gensim (mapping mot <-> id)
dictionary = Dictionary(texts)
print("Taille du vocabulaire avant filtrage :", len(dictionary))

dictionary.filter_extremes(no_below=100, no_above=0.5)
print("Taille du vocabulaire après filtrage :", len(dictionary))

# Corpus au format bag-of-words Gensim
corpus_gensim = [dictionary.doc2bow(text) for text in texts]

Taille du vocabulaire avant filtrage : 89426
Taille du vocabulaire après filtrage : 5589


### D.3 — Modèle LdaModel Gensim

**Objectif :** entraîner un modèle LDA avec Gensim (implémentation différente de celle de scikit-learn) sur ce même corpus.

In [ ]:
gensim_lda = LdaModel(
    corpus=corpus_gensim,
    id2word=dictionary,
    num_topics=10,
    random_state=42,
    passes=5,
)

print("=== Topics LDA (Gensim) ===")
for idx, topic in gensim_lda.print_topics(num_words=10):
    print(f"Topic #{idx}: {topic}")

=== Topics LDA (Gensim) ===
Topic #0: 0.021*"love" + 0.018*"life" + 0.015*"story" + 0.013*"woman" + 0.011*"family" + 0.010*"young" + 0.010*"father" + 0.009*"mother" + 0.008*"girl" + 0.008*"man"
Topic #1: 0.013*"role" + 0.012*"good" + 0.011*"performance" + 0.010*"great" + 0.010*"actor" + 0.010*"cast" + 0.009*"well" + 0.009*"play" + 0.009*"comedy" + 0.009*"best"
Topic #2: 0.017*"see" + 0.016*"like" + 0.015*"really" + 0.015*"time" + 0.014*"good" + 0.013*"great" + 0.012*"would" + 0.012*"think" + 0.010*"story" + 0.010*"watch"
Topic #3: 0.024*"horror" + 0.012*"like" + 0.011*"effect" + 0.010*"good" + 0.010*"original" + 0.008*"monster" + 0.008*"special" + 0.007*"fan" + 0.006*"fun" + 0.006*"little"
Topic #4: 0.013*"character" + 0.009*"story" + 0.007*"scene" + 0.006*"director" + 0.005*"work" + 0.005*"much" + 0.005*"make" + 0.005*"way" + 0.005*"time" + 0.004*"even"
Topic #5: 0.016*"war" + 0.009*"american" + 0.008*"world" + 0.006*"time" + 0.006*"would" + 0.006*"year" + 0.006*"soldier" + 0.006*"peo

### D.4 — Comparaison scikit-learn vs Gensim et impact du prétraitement

**Recoupement avec la Partie B :** on retrouve des topics très proches entre les deux implémentations : guerre/soldat (Gensim #5 ↔ sklearn #9), horreur/effets (Gensim #3 ↔ sklearn #5), séries/école/enfants (Gensim #8 ↔ sklearn #4), performance d'acteurs (Gensim #1 ↔ sklearn #7). Cela confirme que ces thèmes sont des signaux robustes du corpus, indépendants de l'implémentation.

**Un topic nouveau apparaît avec Gensim :** #6, centré sur *song, music, musical, dance, rock* — une thématique **comédies musicales** que ni LDA-sklearn ni NMF n'avaient isolée aussi nettement. Cela vient du filtrage de vocabulaire différent (`no_below=100, no_above=0.5` contre `max_features=5000`) : le vocabulaire retenu n'est pas exactement le même (5 589 mots contre 5 000, sélectionnés selon des critères différents), ce qui peut mettre en avant ou au contraire diluer certains sous-thèmes plus rares.

**Impact du prétraitement observé :** le filtrage `filter_extremes` de Gensim est beaucoup plus agressif sur le vocabulaire brut : il passe de **89 426** tokens uniques à **5 589** (soit −93,7 %), contre un plafond fixe de 5 000 pour CountVectorizer/TfidfVectorizer. Malgré cette différence de méthode de filtrage, les tailles de vocabulaire finales restent comparables (5 589 vs 5 000), et les topics obtenus se recoupent largement — preuve qu'un filtrage raisonnable par fréquence (que ce soit par seuil absolu ou par top-N) capture l'essentiel du signal thématique du corpus. Un filtrage trop laxiste (garder les 89 426 mots) aurait au contraire noyé les modèles dans du bruit (fautes de frappe, noms propres rares, mots à occurrence unique).

---
## Synthèse : LDA vs NMF (5 points)

1. **Nature du modèle** — LDA est un modèle génératif probabiliste (bayésien) ; NMF est une factorisation matricielle déterministe basée sur l'optimisation (minimisation d'une erreur de reconstruction).
2. **Entrée privilégiée** — LDA fonctionne bien sur des comptes bruts (CountVectorizer/Bag of Words) ; NMF donne de meilleurs résultats sur une représentation TF-IDF, qui pondère déjà l'importance des mots. Nos résultats le confirment : NMF sur TF-IDF a produit des topics plus disjoints que LDA sur BoW.
3. **Interprétabilité des topics** — sur notre corpus, NMF a clairement produit des topics plus nets et thématiquement disjoints (séries TV, horreur/zombie, adaptations littéraires, réalisation) alors que LDA a mélangé plusieurs topics autour du vocabulaire évaluatif générique des critiques (*movie, like, good, story*).
4. **Passage à l'échelle et vitesse** — sur nos 50 000 documents, LDA (mode `online`) a mis nettement plus de temps à converger que NMF ; NMF a été l'option la plus rapide ici pour un nombre de topics comparable.
5. **Cadre théorique et incertitude** — LDA offre un cadre probabiliste avec incertitude quantifiable (utile pour l'inférence sur de nouveaux documents, via la distribution topic-document) ; NMF n'offre pas de mesure d'incertitude, mais reste une option robuste et rapide pour une exploration thématique exploratoire, comme observé dans ce TP.